# 16 — FINAL Submission from Locked Pipeline

This notebook performs **no model selection and no tuning**.

Locked pipeline:
- DenseNet121
- 224×224
- grayscale → RGB
- seed 2026 ensemble weight = 0.70
- seed 3407 ensemble weight = 0.30
- each seed prediction = mean of its 5 fold checkpoints
- threshold = 0.576
- TTA = OFF

Outputs:
1. one self-contained `model.pkl` containing all 10 checkpoint state_dicts + inference configuration
2. `<TEAM_ID>.csv` with exactly `Image_id,Prediction`
3. run summary + optional ZIP containing model, CSV and this notebook

**Important:** set `TEST_DIR` and `TEAM_ID` correctly in the config cell before the final competition run. Test labels are never read.


In [ ]:
print('>>> CELL 1 START: final config', flush=True)
import os,sys,json,pickle,zipfile,time,gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image,ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models

ROOT=Path(r'D:\medicine')
LOCK_PATH=ROOT/'cv_outputs'/'final_ensemble_lock'/'locked_pipeline.json'
OUTPUT_DIR=ROOT/'final_submission'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

# ===== EDIT ONLY THESE TWO IF NEEDED =====
TEAM_ID='team01'
TEST_DIR=None
# Example:
# TEST_DIR=Path(r'D:\medicine\test')
# ==========================================

VALID_EXTENSIONS={'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
BATCH_SIZE=16

print('Python:',sys.version.split()[0],flush=True)
print('PyTorch:',torch.__version__,flush=True)
print('Device:',DEVICE,flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0),flush=True)


In [ ]:
print('>>> CELL 2 START: load and verify locked pipeline', flush=True)
with open(LOCK_PATH,'r',encoding='utf-8') as f:
    lock=json.load(f)

assert lock['status']=='LOCKED_BEFORE_HOLDOUT'
assert lock['architecture']=='DenseNet121'
assert lock['image_size']==224
assert lock['preprocessing']=='force_grayscale_then_rgb'
assert lock['tta'] is False
assert abs(sum(float(x['weight']) for x in lock['models'])-1.0)<1e-9
assert len(lock['models'])==2

print('Locked threshold:',lock['threshold'],flush=True)
print('Locked models:',[(x['name'],x['weight']) for x in lock['models']],flush=True)


In [ ]:
print('>>> CELL 3 START: discover unlabeled official test images', flush=True)

def image_paths(root):
    root=Path(root)
    return sorted(
        [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS],
        key=lambda p:str(p).lower()
    )

if TEST_DIR is None:
    candidates=[
        ROOT/'test',
        ROOT/'Test',
        ROOT/'TEST',
        ROOT/'data'/'test',
        ROOT/'data'/'Test',
        ROOT/'dataset'/'test',
        ROOT/'dataset'/'Test',
        ROOT/'chest_xray'/'test',
    ]
    found=[p for p in candidates if p.is_dir() and len(image_paths(p))>0]
    if len(found)==1:
        TEST_DIR=found[0]
    elif len(found)==0:
        raise FileNotFoundError(
            'Official test directory was not auto-detected. Set TEST_DIR in CELL 1 to the organizer-provided unlabeled test folder.'
        )
    else:
        raise RuntimeError(
            'Multiple candidate test directories found: '+str(found)+'. Set TEST_DIR explicitly in CELL 1.'
        )

TEST_DIR=Path(TEST_DIR)
test_paths=image_paths(TEST_DIR)

if not test_paths:
    raise RuntimeError(f'No images found under TEST_DIR={TEST_DIR}')

# We use only paths. No label file/folder is read.
ids=[p.name for p in test_paths]
if len(ids)!=len(set(ids)):
    dup=pd.Series(ids).value_counts()
    dup=dup[dup>1].index.tolist()[:10]
    raise RuntimeError(f'Duplicate base filenames in test set; Image_id would be ambiguous. Examples: {dup}')

print('TEST_DIR:',TEST_DIR,flush=True)
print('Test images:',len(test_paths),flush=True)
print('First 5 Image_id:',ids[:5],flush=True)


In [ ]:
print('>>> CELL 4 START: build ONE self-contained model.pkl', flush=True)

bundle={
    'format':'medical_challenge_2026_locked_ensemble_v1',
    'architecture':'DenseNet121',
    'class_names':['Normal','Pneumonia'],
    'positive_class':'Pneumonia',
    'image_size':int(lock['image_size']),
    'preprocessing':'force_grayscale_then_rgb_pad_square_resize_bilinear',
    'mean':[0.485,0.456,0.406],
    'std':[0.229,0.224,0.225],
    'threshold':float(lock['threshold']),
    'tta':False,
    'seed_groups':[],
    'source_lock':lock,
    'torch_version':torch.__version__,
}

for spec in lock['models']:
    ckpt_dir=Path(spec['checkpoint_dir'])
    group={
        'name':spec['name'],
        'ensemble_weight':float(spec['weight']),
        'fold_state_dicts':[],
        'fold_best_epochs':[],
    }

    print('Packaging:',spec['name'],flush=True)
    for fold in range(5):
        p=ckpt_dir/f'fold{fold}_best.pt'
        if not p.exists():
            raise FileNotFoundError(f'Missing locked checkpoint: {p}')
        ck=torch.load(p,map_location='cpu')
        state={k:v.detach().cpu().clone() for k,v in ck['state_dict'].items()}
        group['fold_state_dicts'].append(state)
        group['fold_best_epochs'].append(int(ck.get('best_epoch',-1)))
        print(f'  fold {fold}: loaded',flush=True)
        del ck,state

    bundle['seed_groups'].append(group)

assert len(bundle['seed_groups'])==2
assert all(len(g['fold_state_dicts'])==5 for g in bundle['seed_groups'])

MODEL_PKL=OUTPUT_DIR/'model.pkl'
with open(MODEL_PKL,'wb') as f:
    pickle.dump(bundle,f,protocol=pickle.HIGHEST_PROTOCOL)

print('model.pkl:',MODEL_PKL,flush=True)
print('Size MiB:',round(MODEL_PKL.stat().st_size/1024**2,2),flush=True)


In [ ]:
print('>>> CELL 5 START: reload model.pkl and build deterministic test dataset', flush=True)
with open(MODEL_PKL,'rb') as f:
    restored=pickle.load(f)

assert restored['format']=='medical_challenge_2026_locked_ensemble_v1'
assert restored['class_names']==['Normal','Pneumonia']
assert restored['tta'] is False

IMAGE_SIZE=int(restored['image_size'])
THRESHOLD=float(restored['threshold'])
MEAN=restored['mean']
STD=restored['std']

class PadToSquare:
    def __call__(self,img):
        w,h=img.size
        side=max(w,h)
        left=(side-w)//2; right=side-w-left
        top=(side-h)//2; bottom=side-h-top
        return ImageOps.expand(img,border=(left,top,right,bottom),fill=0)

eval_tf=transforms.Compose([
    PadToSquare(),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE),interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

class TestDataset(Dataset):
    def __init__(self,paths):
        self.paths=list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        p=self.paths[i]
        with Image.open(p) as im:
            x=eval_tf(im.convert('L').convert('RGB'))
        return x,p.name

test_loader=DataLoader(
    TestDataset(test_paths),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

def make_model():
    m=models.densenet121(weights=None)
    m.classifier=nn.Linear(m.classifier.in_features,2)
    return m.to(DEVICE)

@torch.no_grad()
def infer_state_dict(state_dict):
    model=make_model()
    model.load_state_dict(state_dict,strict=True)
    model.eval()
    probs=[];names=[]
    for x,n in test_loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x)
        p=torch.softmax(logits.float(),dim=1)[:,1]
        probs.extend(p.cpu().numpy())
        names.extend(list(n))
    del model
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()
    return np.asarray(probs,dtype=float),names

print('Reload model.pkl: PASS',flush=True)


In [ ]:
print('>>> CELL 6 START: FINAL test inference from reloaded model.pkl', flush=True)

seed_probs={}
names_ref=None

for group in restored['seed_groups']:
    fold_probs=[]
    print('\nSeed group:',group['name'],flush=True)

    for fold,state in enumerate(group['fold_state_dicts']):
        t0=time.time()
        p,names=infer_state_dict(state)
        if names_ref is None:
            names_ref=names
        else:
            assert names_ref==names
        fold_probs.append(p)
        print(f'  fold {fold}/4 inference: {time.time()-t0:.1f}s',flush=True)

    seed_probs[group['name']]=np.mean(np.stack(fold_probs,axis=1),axis=1)

final_prob=np.zeros(len(test_paths),dtype=float)
for group in restored['seed_groups']:
    final_prob += float(group['ensemble_weight'])*seed_probs[group['name']]

assert len(final_prob)==len(test_paths)
assert np.isfinite(final_prob).all()
assert names_ref==[p.name for p in test_paths]

pred_idx=(final_prob>=THRESHOLD).astype(int)
class_names=restored['class_names']
pred_label=[class_names[i] for i in pred_idx]

submission=pd.DataFrame({
    'Image_id':names_ref,
    'Prediction':pred_label,
})

assert submission.columns.tolist()==['Image_id','Prediction']
assert len(submission)==len(test_paths)
assert submission.Image_id.is_unique
assert set(submission.Prediction).issubset({'Normal','Pneumonia'})

CSV_PATH=OUTPUT_DIR/f'{TEAM_ID}.csv'
submission.to_csv(CSV_PATH,index=False)

# Round-trip exact check.
roundtrip=pd.read_csv(CSV_PATH)
pd.testing.assert_frame_equal(submission,roundtrip)

print('\nFINAL CSV:',CSV_PATH,flush=True)
print('Rows:',len(submission),flush=True)
print('Prediction counts:',submission.Prediction.value_counts().to_dict(),flush=True)
display(submission.head(10))


In [ ]:
print('>>> CELL 7 START: final reproducibility checks', flush=True)

# Store probabilities separately for audit; competition CSV remains exactly two columns.
audit=pd.DataFrame({'Image_id':names_ref,'p_Pneumonia':final_prob,'Prediction':pred_label})
AUDIT_PATH=OUTPUT_DIR/'test_probability_audit.csv'
audit.to_csv(AUDIT_PATH,index=False)

summary={
    'status':'FINAL_LOCKED_SUBMISSION',
    'team_id':TEAM_ID,
    'test_dir':str(TEST_DIR),
    'n_test':len(test_paths),
    'architecture':restored['architecture'],
    'image_size':restored['image_size'],
    'preprocessing':restored['preprocessing'],
    'threshold':THRESHOLD,
    'tta':False,
    'model_groups':[
        {
            'name':g['name'],
            'weight':g['ensemble_weight'],
            'n_fold_models':len(g['fold_state_dicts']),
            'fold_best_epochs':g['fold_best_epochs'],
        }
        for g in restored['seed_groups']
    ],
    'prediction_counts':submission.Prediction.value_counts().to_dict(),
    'model_pkl':MODEL_PKL.name,
    'submission_csv':CSV_PATH.name,
    'holdout_was_not_used_for_retuning':True,
}

SUMMARY_PATH=OUTPUT_DIR/'final_submission_summary.json'
with open(SUMMARY_PATH,'w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2,ensure_ascii=False)

print(json.dumps(summary,indent=2,ensure_ascii=False),flush=True)


In [ ]:
print('>>> CELL 8 START: optional submission ZIP', flush=True)

NOTEBOOK_PATH=ROOT/'notebooks'/'16_final_submission_locked_pipeline.ipynb'
ZIP_PATH=OUTPUT_DIR/f'{TEAM_ID}_submission.zip'

if NOTEBOOK_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(MODEL_PKL,arcname='model.pkl')
        z.write(CSV_PATH,arcname=CSV_PATH.name)
        z.write(NOTEBOOK_PATH,arcname=NOTEBOOK_PATH.name)

    with zipfile.ZipFile(ZIP_PATH,'r') as z:
        names=z.namelist()
        assert 'model.pkl' in names
        assert CSV_PATH.name in names
        assert NOTEBOOK_PATH.name in names

    print('ZIP ready:',ZIP_PATH,flush=True)
    print('ZIP contents:',names,flush=True)
else:
    print('Notebook source file not found at:',NOTEBOOK_PATH,flush=True)
    print('Save/download this notebook, then zip it with model.pkl and the CSV without editing the CSV.',flush=True)


## Final checklist

Before submitting:
- Confirm `TEAM_ID` matches the organizer's required filename.
- Confirm `TEST_DIR` is the official unlabeled Test set.
- Confirm CSV has exactly `Image_id,Prediction`.
- Do not edit any prediction manually.
- Do not retune the locked threshold/weights using holdout or hidden test feedback.
- Keep `test_probability_audit.csv` for your own audit; do not include it unless organizers ask.
